# Useful Notebook: Generate Custom ROC and PRC Plots
**This notebook will allow users to generate (1) ROC and PRC plots for each algorithm (over all CV partitions) if this function was previously turned off in the pipeline, (2) all ROC and PRC plots with the legend inside the plot rather than to the upper right, and (3) allow code-savy users to easily modify this notebook to regenerate these plots to their own specifications.**

*This notebook is designed to run after having run STREAMLINE (at least phases 1-6) and will use the files from a specific STREAMLINE experiment folder, as well as save new output files to that same folder.*

***
## Notebook Details
Generates custom ROC and PRC plots to provide a way to generate modified versions of these plots without rerunning or directly editing the code in the original pipeline. 

When run, 'as-is' this notebook will regenerate all ROC and PRC plots putting the legend inside of the plots: (1) in the upper right hand corner for PRC plots, and (2) in the lower right hand corner for ROC plots. However in some (maybe all cases) the legend will then obstruct the curves themselves. This simple customization of the plots is meant as an example. Users are welcome to modify the plot cells at the bottom of this notebook to further tweak the appearance of the respective plots. These will be saved in the same location within the 'experiment folder' as the original ROC and PRC plots, but with a filename modified by `name_modifier` below.
 

***
## Notebook Run Parameters
* This notbook has been set up to run 'as-is' on the experiment folder generated when running the demo of STREAMLINE in any mode (if no run parameters were changed). 
* If you have run STREAMLINE on different target data or saved the experiment to some other folder outside of STREAMLINE, you need to edit `experiment_path` below to point to the respective experiment folder.

In [ ]:
experiment_path = "/Users/harshbandhey/Local/Cedars/Urbslab/STREAMLINEv3New/test/out_full_pipeline/DemoExp" # path the target experiment folder 
targetDataName = None # 'None' if user wants to generate visualizations for all analyzed datasets, otherwise give a (str) list of target dataset names
algorithms = [] #use empty list if user wishes to plot feature importance for all modeling algorithms that were run in pipeline.
name_modifier = '_New' # Modifies standard plot filenames to avoid overwriting originals.
legend_inside_plot = True #place legend inside plot, other wise placed outside on upper right hand corner.
plot_ROC = True # For each algorithm plot ROC curve - including lines for each trained CV model.
plot_PRC = True # For each algorithm plot PRC curve - including lines for each trained CV model.
plot_meta_ROC = True #Generate ROC summarizing average ROC curves (all cvs) for each algorithm
plot_meta_PRC = True #Generate PRC summarizing average ROC curves (all cvs) for each algorithm

***
## Housekeeping
### Import Packages

In [ ]:
import os
import json
import pandas as pd
import pickle
from statistics import mean,stdev
import matplotlib.pyplot as plt
from matplotlib import rc
import numpy as np
from scipy import stats
interp = np.interp

import warnings
warnings.filterwarnings('ignore')

# Jupyter Notebook Hack: This code ensures that the results of multiple commands within a given cell are all displayed, rather than just the last. 
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

### Automatically detect data folder names

In [ ]:
# Get dataset paths for all completed dataset analyses in experiment folder
experiment_name = experiment_path.split('/')[-1]
remove_list = {
    '.DS_Store', 'metadata.pickle', 'metadata.csv', 'algInfo.pickle',
    'DatasetComparisons', 'jobs', 'jobsCompleted', 'logs', 'KeyFileCopy', 'dask_logs',
    'reporting', 'reporting_replication', 'run_params.pickle', 'runtime',
    experiment_name + '_STREAMLINE_Report.pdf'
}

datasets = []
for d in sorted(os.listdir(experiment_path)):
    dpath = os.path.join(experiment_path, d)
    if d in remove_list or not os.path.isdir(dpath):
        continue
    has_exploratory = os.path.isdir(os.path.join(dpath, 'exploratory'))
    has_model_data = os.path.isdir(os.path.join(dpath, 'model_evaluation')) or os.path.isdir(os.path.join(dpath, 'models'))
    if has_exploratory and has_model_data:
        datasets.append(d)

print("Analyzed Datasets: " + str(datasets))

### Load other necessary parameters

In [ ]:
# Unpickle metadata from previous phase
file = open(experiment_path+'/'+"metadata.pickle", 'rb')
metadata = pickle.load(file)
file.close()
# Load variables specified earlier in the pipeline from metadata
outcome_label = metadata.get('Outcome Label', metadata.get('Class Label', 'Class'))
instance_label = metadata['Instance Label']
cv_partitions = int(metadata['CV Partitions'])

requested_algorithms = list(algorithms) if isinstance(algorithms, list) else []

# Unpickle algorithm information from previous phase
alg_info_path = os.path.join(experiment_path, "algInfo.pickle")
if os.path.exists(alg_info_path):
    with open(alg_info_path, "rb") as file:
        algInfo = pickle.load(file)
else:
    algInfo = {}

algorithms = []
abbrev = {}
colors = {}
for key, value in algInfo.items():
    if isinstance(value, (list, tuple)) and len(value) > 0 and bool(value[0]):
        algorithms.append(key)
        abbrev[key] = value[1] if len(value) > 1 else key
        colors[key] = value[2] if len(value) > 2 else None

# Fallback: infer algorithms from current output layout
if not algorithms:
    inferred = set()
    scan_datasets = [d for d in datasets if os.path.isdir(os.path.join(experiment_path, d))]
    for ds_name in scan_datasets:
        model_dir = os.path.join(experiment_path, ds_name, "models", "pickledModels")
        if os.path.isdir(model_dir):
            for fname in os.listdir(model_dir):
                if fname.endswith('.pickle') and '_' in fname:
                    inferred.add(fname.rsplit('_', 1)[0])
        metric_dir = os.path.join(experiment_path, ds_name, "model_evaluation", "metrics_by_cv")
        if os.path.isdir(metric_dir):
            for fname in os.listdir(metric_dir):
                if fname.endswith('.json') and '_CV_' in fname:
                    inferred.add(fname.split('_CV_')[0])
    for abr in sorted(inferred):
        algorithms.append(abr)
        abbrev[abr] = abr
        colors[abr] = None

if requested_algorithms:
    req = set(requested_algorithms)
    filtered = [a for a in algorithms if a in req or abbrev.get(a) in req]
    if filtered:
        algorithms = filtered

palette = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#8c564b", "#e377c2", "#7f7f7f", "#bcbd22", "#17becf"]
for idx, key in enumerate(algorithms):
    if colors.get(key) is None:
        colors[key] = palette[idx % len(palette)]

print(algorithms)

## Define Necessary Methods

In [ ]:
def primaryStats(algorithms,original_headers,cv_partitions,full_path,data_name,instance_label,outcome_label,abbrev,colors,plot_ROC,plot_PRC,name_modifier,legend_inside_plot):
    """Combine classification metrics and curves from current JSON outputs."""

    def _metric_val(metrics_map, keys, default=np.nan):
        for k in keys:
            if k in metrics_map and metrics_map[k] is not None:
                return metrics_map[k]
        return default

    def _load_curve(path):
        if not os.path.exists(path):
            return None
        with open(path, 'r') as f:
            payload = json.load(f)
        if isinstance(payload, dict):
            if 'micro' in payload and isinstance(payload['micro'], dict):
                return payload['micro']
            if 'binary' in payload and isinstance(payload['binary'], dict):
                return payload['binary']
            if 'fpr' in payload or 'precision' in payload:
                return payload
        return None

    def _safe_interp(x_new, x, y):
        x = np.asarray(x, dtype=float)
        y = np.asarray(y, dtype=float)
        if len(x) == 0 or len(y) == 0:
            return np.zeros_like(x_new)
        order = np.argsort(x)
        x = x[order]
        y = y[order]
        x_unique, idx = np.unique(x, return_index=True)
        y_unique = y[idx]
        if len(x_unique) == 1:
            return np.full_like(x_new, y_unique[0], dtype=float)
        return interp(x_new, x_unique, y_unique)

    result_table = []
    metric_dict = {}
    for algorithm in algorithms:
        alg_result_table = []
        s_bac, s_ac, s_f1, s_re, s_sp, s_pr = [], [], [], [], [], []
        s_tp, s_tn, s_fp, s_fn, s_npv, s_lrp, s_lrm = [], [], [], [], [], [], []
        tprs, aucs = [], []
        mean_fpr = np.linspace(0, 1, 100)
        mean_recall = np.linspace(0, 1, 100)
        precs, praucs, aveprecs = [], [], []

        for cvCount in range(0,cv_partitions):
            metrics_file = full_path+'/model_evaluation/metrics_by_cv/'+abbrev[algorithm]+'_CV_'+str(cvCount)+'.json'
            roc_file = full_path+'/model_evaluation/curves_by_cv/'+abbrev[algorithm]+'_CV_'+str(cvCount)+'_roc.json'
            prc_file = full_path+'/model_evaluation/curves_by_cv/'+abbrev[algorithm]+'_CV_'+str(cvCount)+'_prc.json'

            if not os.path.exists(metrics_file):
                continue

            with open(metrics_file, 'r') as f:
                metrics_payload = json.load(f)
            metrics_map = metrics_payload.get('metrics', {})

            roc_curve = _load_curve(roc_file)
            prc_curve = _load_curve(prc_file)
            if roc_curve is None or prc_curve is None:
                continue

            fpr = np.asarray(roc_curve.get('fpr', []), dtype=float)
            tpr = np.asarray(roc_curve.get('tpr', []), dtype=float)
            prec = np.asarray(prc_curve.get('precision', []), dtype=float)
            recall = np.asarray(prc_curve.get('recall', []), dtype=float)
            if len(fpr) < 2 or len(tpr) < 2 or len(prec) < 2 or len(recall) < 2:
                continue

            roc_auc = float(_metric_val(metrics_map, ['roc_auc_micro', 'roc_auc', 'roc_auc_macro'], roc_curve.get('auc', np.nan)))
            prec_rec_auc = float(_metric_val(metrics_map, ['average_precision_micro', 'average_precision', 'average_precision_macro'], prc_curve.get('pr_auc', np.nan)))
            ave_prec = float(_metric_val(metrics_map, ['average_precision_micro', 'average_precision', 'average_precision_macro'], prc_curve.get('aps', np.nan)))

            s_bac.append(float(_metric_val(metrics_map, ['balanced_accuracy'], np.nan)))
            s_ac.append(float(_metric_val(metrics_map, ['accuracy', 'f1_micro'], np.nan)))
            s_f1.append(float(_metric_val(metrics_map, ['f1', 'f1_macro', 'f1_micro'], np.nan)))
            s_re.append(float(_metric_val(metrics_map, ['recall', 'recall_macro', 'recall_micro', 'sensitivity'], np.nan)))
            s_sp.append(np.nan)
            s_pr.append(float(_metric_val(metrics_map, ['precision', 'precision_macro', 'precision_micro'], np.nan)))
            s_tp.append(np.nan)
            s_tn.append(np.nan)
            s_fp.append(np.nan)
            s_fn.append(np.nan)
            s_npv.append(np.nan)
            s_lrp.append(np.nan)
            s_lrm.append(np.nan)

            alg_result_table.append([fpr, tpr, roc_auc, prec, recall, prec_rec_auc, ave_prec])
            tprs.append(_safe_interp(mean_fpr, fpr, tpr))
            tprs[-1][0] = 0.0
            aucs.append(roc_auc)

            precs.append(_safe_interp(mean_recall, recall, prec))
            praucs.append(prec_rec_auc)
            aveprecs.append(ave_prec)

        if not aucs:
            print('Skipping algorithm with no usable CV metrics: ' + str(algorithm))
            continue

        print(algorithm)
        mean_tpr = np.mean(tprs, axis=0)
        mean_tpr[-1] = 1.0
        mean_auc = np.mean(aucs)

        if plot_ROC:
            plt.rcParams["figure.figsize"] = (6,6)
            for i in range(len(alg_result_table)):
                plt.plot(alg_result_table[i][0], alg_result_table[i][1], lw=1, alpha=0.3,label='ROC fold %d (AUC = %0.3f)' % (i, alg_result_table[i][2]))
            plt.plot([0, 1], [0, 1], linestyle='--', lw=2, color='r',label='No-Skill', alpha=.8)
            std_auc = np.std(aucs)
            plt.plot(mean_fpr, mean_tpr, color=colors[algorithm],label=r'Mean ROC (AUC = %0.3f $\pm$ %0.3f)' % (mean_auc, std_auc),lw=2, alpha=.8)
            std_tpr = np.std(tprs, axis=0)
            tprs_upper = np.minimum(mean_tpr + std_tpr, 1)
            tprs_lower = np.maximum(mean_tpr - std_tpr, 0)
            plt.fill_between(mean_fpr, tprs_lower, tprs_upper, color='grey', alpha=.2,label=r'$\pm$ 1 std. dev.')
            plt.xlim([-0.05, 1.05])
            plt.ylim([-0.05, 1.05])
            plt.title(str(algorithm))
            plt.xlabel('False Positive Rate')
            plt.ylabel('True Positive Rate')
            if legend_inside_plot:
                plt.legend(loc="lower right")
            else:
                plt.legend(loc="upper left", bbox_to_anchor=(1.01,1))
            plt.savefig(full_path+'/model_evaluation/'+abbrev[algorithm]+"_ROC"+name_modifier+".png", bbox_inches="tight")
            plt.show()

        mean_prec = np.mean(precs, axis=0)
        mean_pr_auc = np.mean(praucs)
        if plot_PRC:
            plt.rcParams["figure.figsize"] = (6,6)
            for i in range(len(alg_result_table)):
                plt.plot(alg_result_table[i][4], alg_result_table[i][3], lw=1, alpha=0.3, label='PRC fold %d (AUC = %0.3f)' % (i, alg_result_table[i][5]))
            test = pd.read_csv(full_path + '/CVDatasets/' + data_name + '_CV_0_Test.csv')
            testY = test[outcome_label].values
            uniq = np.unique(testY)
            noskill = 0.5
            if len(uniq) == 2:
                noskill = float(len(testY[testY == 1]) / len(testY))
            plt.plot([0, 1], [noskill, noskill], color='orange', linestyle='--', label='No-Skill', alpha=.8)
            std_pr_auc = np.std(praucs)
            plt.plot(mean_recall, mean_prec, color=colors[algorithm],label=r'Mean PRC (AUC = %0.3f $\pm$ %0.3f)' % (mean_pr_auc, std_pr_auc),lw=2, alpha=.8)
            std_prec = np.std(precs, axis=0)
            precs_upper = np.minimum(mean_prec + std_prec, 1)
            precs_lower = np.maximum(mean_prec - std_prec, 0)
            plt.fill_between(mean_recall, precs_lower, precs_upper, color='grey', alpha=.2,label=r'$\pm$ 1 std. dev.')
            plt.xlim([-0.05, 1.05])
            plt.ylim([-0.05, 1.05])
            plt.title(str(algorithm))
            plt.xlabel('Recall (Sensitivity)')
            plt.ylabel('Precision (PPV)')
            if legend_inside_plot:
                plt.legend(loc="upper right")
            else:
                plt.legend(loc="upper left", bbox_to_anchor=(1.01,1))
            plt.savefig(full_path+'/model_evaluation/'+abbrev[algorithm]+"_PRC"+name_modifier+".png", bbox_inches="tight")
            plt.show()

        results = {
            'Balanced Accuracy': s_bac, 'Accuracy': s_ac, 'F1 Score': s_f1,
            'Sensitivity (Recall)': s_re, 'Specificity': s_sp, 'Precision (PPV)': s_pr,
            'TP': s_tp, 'TN': s_tn, 'FP': s_fp, 'FN': s_fn,
            'NPV': s_npv, 'LR+': s_lrp, 'LR-': s_lrm,
            'ROC AUC': aucs,'PRC AUC': praucs, 'PRC APS': aveprecs
        }
        metric_dict[algorithm] = results

        mean_ave_prec = np.mean(aveprecs)
        result_dict = {
            'algorithm':algorithm,'fpr':mean_fpr, 'tpr':mean_tpr, 'auc':mean_auc,
            'prec':mean_prec, 'recall':mean_recall, 'pr_auc':mean_pr_auc, 'ave_prec':mean_ave_prec
        }
        result_table.append(result_dict)

    if result_table:
        result_table = pd.DataFrame.from_dict(result_table)
        result_table.set_index('algorithm',inplace=True)
    else:
        result_table = pd.DataFrame()

    return result_table,metric_dict

In [ ]:
def doPlotROC(result_table,colors,full_path,name_modifier,legend_inside_plot):
    """ Generate ROC plot comparing average ML algorithm performance (over all CV training/testing sets)"""
    if result_table is None or result_table.empty:
        print('No ROC data available to plot.')
        return
    for i in result_table.index:
        plt.plot(result_table.loc[i]['fpr'],result_table.loc[i]['tpr'], color=colors[i],label="{}, AUC={:.3f}".format(i, result_table.loc[i]['auc']))
    plt.rcParams["figure.figsize"] = (6,6)
    plt.plot([0, 1], [0, 1], color='orange', linestyle='--', label='No-Skill', alpha=.8)
    plt.xticks(np.arange(0.0, 1.1, step=0.1))
    plt.xlabel("False Positive Rate", fontsize=15)
    plt.yticks(np.arange(0.0, 1.1, step=0.1))
    plt.ylabel("True Positive Rate", fontsize=15)
    if legend_inside_plot:
        plt.legend(loc="lower right")
    else:
        plt.legend(loc="upper left", bbox_to_anchor=(1.01,1))
    plt.savefig(full_path+'/model_evaluation/Summary_ROC'+name_modifier+'.png', bbox_inches="tight")
    plt.show()

In [ ]:
def doPlotPRC(result_table,colors,full_path,data_name,instance_label,outcome_label,name_modifier,legend_inside_plot):
    """ Generate PRC plot comparing average ML algorithm performance (over all CV training/testing sets)"""
    if result_table is None or result_table.empty:
        print('No PRC data available to plot.')
        return
    for i in result_table.index:
        plt.plot(result_table.loc[i]['recall'],result_table.loc[i]['prec'], color=colors[i],label="{}, AUC={:.3f}, APS={:.3f}".format(i, result_table.loc[i]['pr_auc'],result_table.loc[i]['ave_prec']))

    test = pd.read_csv(full_path+'/CVDatasets/'+data_name+'_CV_0_Test.csv')
    if instance_label != 'None' and instance_label in test.columns:
        test = test.drop(instance_label, axis=1)
    testY = test[outcome_label].values
    uniq = np.unique(testY)
    noskill = 0.5
    if len(uniq) == 2:
        noskill = len(testY[testY == 1]) / len(testY)
    plt.plot([0, 1], [noskill, noskill], color='orange', linestyle='--',label='No-Skill', alpha=.8)
    plt.xticks(np.arange(0.0, 1.1, step=0.1))
    plt.xlabel("Recall (Sensitivity)", fontsize=15)
    plt.yticks(np.arange(0.0, 1.1, step=0.1))
    plt.ylabel("Precision (PPV)", fontsize=15)
    if legend_inside_plot:
        plt.legend(loc="upper right")
    else:
        plt.legend(loc="upper left", bbox_to_anchor=(1.01,1))
    plt.savefig(full_path+'/model_evaluation/Summary_PRC'+name_modifier+'.png', bbox_inches="tight")
    plt.show()

## Generate all ROC and PRC Plots

In [ ]:
if targetDataName not in (None, 'None'):
    datasets = [d for d in datasets if d == targetDataName]

if not algorithms:
    raise ValueError("No algorithms discovered. Check experiment path and model outputs.")

for each in datasets: #each analyzed dataset to make plots for
    print("---------------------------------------")
    print("Dataset: "+str(each))
    print("---------------------------------------")
    full_path = experiment_path+'/'+each
    original_headers = pd.read_csv(full_path+"/exploratory/ProcessedFeatureNames.csv",sep=',').columns.values.tolist()

    result_table,metric_dict = primaryStats(algorithms,original_headers,cv_partitions,full_path,each,instance_label,outcome_label,abbrev,colors,plot_ROC,plot_PRC,name_modifier,legend_inside_plot)

    if plot_meta_ROC:
        doPlotROC(result_table,colors,full_path,name_modifier,legend_inside_plot)
    if plot_meta_PRC:
        doPlotPRC(result_table,colors,full_path,each,instance_label,outcome_label,name_modifier,legend_inside_plot)